# 03 — Champion Balance Analysis

## Business Question
Which champions are statistically over or underperforming, and is the community banning the right champions?

## What Makes This Rigorous
- **Binomial test** per champion: is win rate significantly different from 50%?
- **Benjamini-Hochberg FDR correction** for multiple testing (138 simultaneous tests)
- **Combined balance score**: win rate + ban rate + pick rate combined metric
- **Spearman correlation**: does community ban priority match actual win rate?
- **Quadrant analysis**: High ban + high win = genuine problem vs perceived problem

In [1]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from config import *
from data_loader import load_matches, load_champion_map, build_champion_stats
from stats_utils import test_win_rate, benjamini_hochberg, spearman_correlation
from plot_utils import set_style, save_plot

set_style()
df = load_matches()
champ_map = load_champion_map()
stats = build_champion_stats(df, champ_map, min_games=MIN_GAMES_CHAMPION)
print(f"Champions analysed: {len(stats)}")

Champions analysed: 138


## 3.1 — Binomial Tests with Multiple Testing Correction

In [2]:
# Run binomial test for each champion
test_results = []
for _, row in stats.iterrows():
    r = test_win_rate(int(row['wins']), int(row['games']), h0_rate=0.5, label=row['champion'])
    test_results.append(r)

tests_df = pd.DataFrame(test_results)

# Apply Benjamini-Hochberg FDR correction (better than Bonferroni for many tests)
p_values = tests_df['p_value'].tolist()
bh_significant = benjamini_hochberg(p_values, alpha=ALPHA)
tests_df['significant_bh'] = bh_significant
tests_df['direction'] = tests_df.apply(
    lambda r: 'Overpowered' if r['observed_pct'] > 50 else 'Underpowered', axis=1)

# Merge back
stats = stats.merge(
    tests_df[['label','p_value','significant_bh','ci_low_pct','ci_high_pct']],
    left_on='champion', right_on='label', how='left'
)

print(f"Champions with significant win rate (BH-corrected): {stats['significant_bh'].sum()}")
print(f"Overpowered (>50%, significant): {((stats['win_rate'] > 50) & stats['significant_bh']).sum()}")
print(f"Underpowered (<50%, significant): {((stats['win_rate'] < 50) & stats['significant_bh']).sum()}")
print("\nTop 10 by win rate (significant only):")
sig_only = stats[stats['significant_bh']].sort_values('win_rate', ascending=False)
print(sig_only[['champion','games','win_rate','ban_rate','ci_low_pct','ci_high_pct']].head(10).to_string(index=False))

Champions with significant win rate (BH-corrected): 46
Overpowered (>50%, significant): 23
Underpowered (<50%, significant): 23

Top 10 by win rate (significant only):
champion  games  win_rate  ban_rate  ci_low_pct  ci_high_pct
   Janna   8691     55.53     41.54       54.48        56.57
    Sona   5429     54.19      1.19       52.86        55.51
  Yorick   1378     53.99      0.98       51.35        56.61
  Rammus   2997     53.85      3.59       52.07        55.63
  Anivia   2252     53.60      1.70       51.53        55.65
  Singed   1425     53.47      0.98       50.88        56.05
   Swain   1494     53.15      0.91       50.61        55.67
 Sejuani   3867     53.12     10.16       51.54        54.69
   Garen   3893     53.10      3.63       51.53        54.66
 Shyvana   2445     53.01      0.75       51.02        54.98


In [3]:
# Win rate chart with CIs — significant champions only
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 15 overpowered
op_champs = stats[(stats['win_rate'] > 50) & stats['significant_bh']].nlargest(15, 'win_rate')
op_sorted = op_champs.sort_values('win_rate', ascending=True)
bars = axes[0].barh(op_sorted['champion'], op_sorted['win_rate'],
                    color=COLORS['green'], edgecolor='white', height=0.7, alpha=0.85)
for i, (_, row) in enumerate(op_sorted.iterrows()):
    axes[0].plot([row['ci_low_pct'], row['ci_high_pct']], [i, i],
                 color='black', linewidth=2, solid_capstyle='round')
    axes[0].text(row['ci_high_pct'] + 0.2, i, f"{row['win_rate']}%",
                va='center', fontsize=9, fontweight='bold')
axes[0].axvline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Win Rate (%)')
axes[0].set_title('Significantly Overpowered Champions\n(BH-corrected, with 95% CI)', color=COLORS['green'])
axes[0].set_xlim(44, 62)

# Bottom 15 underpowered
up_champs = stats[(stats['win_rate'] < 50) & stats['significant_bh']].nsmallest(15, 'win_rate')
up_sorted = up_champs.sort_values('win_rate', ascending=False)
bars2 = axes[1].barh(up_sorted['champion'], up_sorted['win_rate'],
                     color=COLORS['red'], edgecolor='white', height=0.7, alpha=0.85)
for i, (_, row) in enumerate(up_sorted.iterrows()):
    axes[1].plot([row['ci_low_pct'], row['ci_high_pct']], [i, i],
                 color='black', linewidth=2, solid_capstyle='round')
    axes[1].text(row['ci_high_pct'] + 0.2, i, f"{row['win_rate']}%",
                va='center', fontsize=9, fontweight='bold')
axes[1].axvline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Win Rate (%)')
axes[1].set_title('Significantly Underpowered Champions\n(BH-corrected, with 95% CI)', color=COLORS['red'])
axes[1].set_xlim(38, 55)

plt.suptitle('Champion Balance Analysis — Season 9', fontsize=14, fontweight='bold')
save_plot('03a_champion_balance.png')
plt.show()

  Saved -> plots/03a_champion_balance.png


## 3.2 — Ban Rate vs Win Rate Quadrant Analysis

In [4]:
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(
    stats['ban_rate'], stats['win_rate'],
    s=stats['games'] / 25,
    c=stats['win_rate'],
    cmap='RdYlGn', vmin=44, vmax=57,
    alpha=0.75, edgecolors='white', linewidth=0.5
)

# Quadrant lines
ax.axhline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(BAN_RATE_HIGH, color=COLORS['gray'], linestyle='--', linewidth=1.5, alpha=0.7)

# Label notable champions
for _, row in stats.iterrows():
    if row['win_rate'] > 53 or row['win_rate'] < 47 or row['ban_rate'] > 40:
        ax.annotate(row['champion'],
            (row['ban_rate'], row['win_rate']),
            fontsize=8, ha='center', va='bottom',
            xytext=(0, 4), textcoords='offset points')

# Quadrant labels
ax.text(60, 55.5, 'GENUINE PROBLEM\n(High ban + High win)', fontsize=9,
        color=COLORS['red'], fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['red'], alpha=0.8))
ax.text(5, 55.5, 'HIDDEN OP\n(Low ban + High win)', fontsize=9,
        color=COLORS['orange'], fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['orange'], alpha=0.8))
ax.text(60, 44.5, 'PERCEIVED THREAT\n(High ban + Low win)', fontsize=9,
        color=COLORS['blue'], fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['blue'], alpha=0.8))

plt.colorbar(scatter, ax=ax, label='Win Rate (%)')
ax.set_xlabel('Ban Rate (% of games banned)')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Champion Quadrant Analysis\nBan Rate vs Win Rate (bubble size = games played)')
save_plot('03b_ban_vs_win_quadrant.png')
plt.show()

  Saved -> plots/03b_ban_vs_win_quadrant.png


In [5]:
# Spearman correlation between ban rate and win rate
corr_result = spearman_correlation(stats['ban_rate'].values, stats['win_rate'].values,
                                    label='Ban Rate vs Win Rate')
print("=== Spearman Correlation: Ban Rate vs Win Rate ===")
for k, v in corr_result.items():
    print(f"  {k}: {v}")

print("\n=== Interpretation ===")
if corr_result['significant']:
    print(f"There IS a {corr_result['strength']} {corr_result['direction']} correlation between ban rate and win rate.")
    if corr_result['direction'] == 'positive':
        print("Community ban priority ALIGNS with actual win rates — players are banning the right champions.")
    else:
        print("Community ban priority MISALIGNS with actual win rates — players may be banning based on perception, not data.")
else:
    print("No significant correlation — ban decisions appear independent of win rates.")

=== Spearman Correlation: Ban Rate vs Win Rate ===
  label: Ban Rate vs Win Rate
  correlation: 0.1918
  p_value: 0.024222
  significant: True
  strength: negligible
  direction: positive

=== Interpretation ===
There IS a negligible positive correlation between ban rate and win rate.
Community ban priority ALIGNS with actual win rates — players are banning the right champions.


## Summary

| Champion Type | Count | Action |
|---|---|---|
| Statistically overpowered (BH-corrected) | Check output | Consider nerfs |
| Statistically underpowered | Check output | Consider buffs |
| High ban + High win (Genuine Problem) | See quadrant chart | Priority balance target |
| High ban + Low win (Perceived Threat) | See quadrant chart | Community perception issue |

**Key insight:** The Spearman correlation between ban rate and win rate tells us whether the player community's threat assessment is actually data-accurate — a crucial question for balance team communication strategy.